In [ ]:
import numpy as np
import open3d as o3d
from skimage import measure


def bpa_with_multiscale_radii(pcd, base_radius, dists):
    # Geometric stack around base_radius (robust across curvature changes)
    #radii = [base_radius * r for r in (0.9, 1.3, 1.9, 2.7)]
    radii = [np.min(dists), 0.85*np.median(dists), 1.0*np.median(dists), 1.2*np.median(dists), np.max(dists)]
    mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
        pcd, o3d.utility.DoubleVector(radii)
    )
    mesh.remove_duplicated_vertices(); mesh.remove_degenerate_triangles()
    mesh.remove_duplicated_triangles(); mesh.remove_unreferenced_vertices()
    mesh.remove_non_manifold_edges()
    return mesh


# ------------ MAIN ------------
# Load binary voxels → closed mesh via marching cubes
voxels = np.load("../data/sphere.npy")  # 0/1 (nz, ny, nz)
verts, faces, normals, values = measure.marching_cubes(
    voxels, spacing=(1.0, 1.0, 1.0), level=0.5
)
src = o3d.geometry.TriangleMesh(
    o3d.utility.Vector3dVector(verts),
    o3d.utility.Vector3iVector(faces)
)
src.remove_duplicated_vertices(); src.remove_degenerate_triangles()
src.remove_duplicated_triangles(); src.remove_unreferenced_vertices()
src.remove_non_manifold_edges()
src.compute_vertex_normals()
print("loaded & cleaned...")
print("Watertight:", src.is_watertight())
print("Edge-manifold:", src.is_edge_manifold())
print("Vertex-manifold:", src.is_vertex_manifold())

# 1) Uniform resampling control: choose target edge length h
h = 2.0  # tune this knob (units = your spacing units)

surf_area = src.get_surface_area()
print("Surface area:", surf_area)
n_pts = int(2 * surf_area / (np.sqrt(3)/4 * h*h))
pcd = src.sample_points_poisson_disk(n_pts, init_factor=5, pcl=None)
# Robust normals for BPA
pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=2.5*h, max_nn=60)
)
pcd.orient_normals_consistent_tangent_plane(k=50)

# 2) BPA with data-driven radii, densify & retry if needed
attempts = []
mesh_bpa = None
for i, mult in enumerate([1.0, 1.0, 1.0]):  # one densification retry if needed
    if i > 0:
        # densify points (~double) around same h to bridge gaps
        n_pts = int(n_pts * 1.6)
        pcd = src.sample_points_poisson_disk(n_pts, init_factor=5, pcl=None)
        pcd.estimate_normals(
            search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=2.5*h, max_nn=60)
        )
        pcd.orient_normals_consistent_tangent_plane(k=50)

    dists = pcd.compute_nearest_neighbor_distance()
    print("min/mean/med/max dist:", np.min(dists), np.mean(dists), np.median(dists), np.max(dists))
    d = float(np.median(dists))
    base_r = max(1e-6, d * mult)  # avoid zero
    print(i, "Base radius:", base_r, "for approx spacing", d)
    mesh_try = bpa_with_multiscale_radii(pcd, base_r, dists)

    attempts.append((mesh_try, base_r, int(np.asarray(mesh_try.triangles).shape[0])))
    if mesh_try.is_watertight() and mesh_try.is_edge_manifold() and mesh_try.is_vertex_manifold():
        mesh_bpa = mesh_try
        break

if mesh_bpa is not None:
    mesh = mesh_bpa
else:
    print("WARNING: all attempts failed")
    raise RuntimeError("Could not obtain watertight mesh via BPA")

# 4) Non-shrinking smoothing + QA
# vol_before = mesh.get_volume() if mesh.is_watertight() else None
# mesh = mesh.filter_smooth_taubin(number_of_iterations=20)  # mild, non-shrinking
# mesh.orient_triangles()
# mesh.compute_vertex_normals()


print("Watertight:", mesh.is_watertight())
print("Edge-manifold:", mesh.is_edge_manifold())
print("Vertex-manifold:", mesh.is_vertex_manifold())

# Visualize and/or save
# o3d.visualization.draw_geometries([mesh])
# o3d.io.write_triangle_mesh("surface_uniform_watertight.stl", mesh, write_ascii=False)
# print("Saved: surface_uniform_watertight.stl")


loaded & cleaned...
Watertight: True
Edge-manifold: True
Vertex-manifold: True
Surface area: 3467.600256338601
min/mean/med/max dist: 0.6845547226902466 0.7817473723033576 0.7718964295339679 1.1225546090305054
0 Base radius: 0.7718964295339679 for approx spacing 0.7718964295339679
min/mean/med/max dist: 0.5413553345765886 0.6181822883309898 0.6099965828518354 0.855715743476861
1 Base radius: 0.6099965828518354 for approx spacing 0.6099965828518354
min/mean/med/max dist: 0.4298480565593614 0.49056141856603075 0.48413284306349885 0.7289567643400177
2 Base radius: 0.48413284306349885 for approx spacing 0.48413284306349885


RuntimeError: Could not obtain watertight mesh via BPA